# T4D Response V2 - Phase 3 Water Temperature Model

Goal: predict daily `delta_water_temp` from atmospheric forcing with simple station context built from training stations only.

Outputs:
- `t4d.v2.phase3.phase6_scores.csv`
- `t4d.v2.phase3.daily_with_temp_pred.csv`
- `t4d.v2.phase3.phase6_holdout_predictions.csv`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme( style = 'whitegrid' )
pd.set_option( 'display.max_columns', 120 )

phase_dir = Path( '../data/reference/response_v2' )
daily_path = phase_dir / 't4d.v2.phase2.daily_screened.csv'
cohort_path = phase_dir / 't4d.v2.phase2.cohort_keys.csv'

scores_out_path = phase_dir / 't4d.v2.phase3.phase6_scores.csv'
daily_pred_out_path = phase_dir / 't4d.v2.phase3.daily_with_temp_pred.csv'
holdout_pred_out_path = phase_dir / 't4d.v2.phase3.phase6_holdout_predictions.csv'

daily = pd.read_csv( daily_path, parse_dates = [ 'date' ] )
cohort_keys = pd.read_csv( cohort_path )
daily = daily.merge( cohort_keys[ [ 'station_key', 'cohort' ] ], on = [ 'station_key', 'cohort' ], how = 'left' )

In [ ]:
# Build simple features.
# Keep the context honest.
# The station context comes from training stations only.
feature_cols = [ 'air_temp', 'wind_speed', 'precipitation', 'solar_radiation', 'doy_sin', 'doy_cos' ]
feature_cols = [ col for col in feature_cols if col in daily.columns ]

rolling_sources = [ 'air_temp', 'wind_speed', 'precipitation', 'solar_radiation' ]
rolling_sources = [ col for col in rolling_sources if col in daily.columns ]

for source_col in rolling_sources:
    for window in [ 1, 7, 28 ]:
        daily[ f'{ source_col }_r{ window }d' ] = ( 
            daily
            .sort_values( [ 'station_key', 'date' ] )
            .groupby( 'station_key' )[ source_col ]
            .transform( lambda values: values.rolling( window = window, min_periods = 1 ).mean( ) )
        )

model_feature_cols = [ ]
for source_col in rolling_sources:
    for window in [ 1, 7, 28 ]:
        model_feature_cols.append( f'{ source_col }_r{ window }d' )
model_feature_cols += [ col for col in [ 'air_temp', 'wind_speed', 'precipitation', 'solar_radiation', 'doy_sin', 'doy_cos' ] if col in daily.columns ]

train_daily = daily.loc[ daily[ 'cohort' ] == 'train' ].copy( )
holdout_daily = daily.loc[ daily[ 'cohort' ] == 'holdout' ].copy( )

context_cols = [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ]
context_cols = [ col for col in context_cols if col in train_daily.columns ]
context = ( 
    train_daily
    .groupby( [ 'station_key' ], as_index = False )[ context_cols ]
    .mean( )
    .rename( columns = { col: f'ctx_{ col }_mean' for col in context_cols } )
)

daily = daily.merge( context, on = 'station_key', how = 'left' )
train_daily = daily.loc[ daily[ 'cohort' ] == 'train' ].copy( )
holdout_daily = daily.loc[ daily[ 'cohort' ] == 'holdout' ].copy( )

context_feature_cols = [ col for col in daily.columns if col.startswith( 'ctx_' ) ]
all_feature_cols = model_feature_cols + context_feature_cols + [ col for col in [ 'water_temp_baseline', 'salinity_baseline', 'oxygen_baseline', 'ph_baseline', 'depth_baseline' ] if col in daily.columns ]
all_feature_cols = list( dict.fromkeys( all_feature_cols ) )

target_col = 'delta_water_temp'
train_model = train_daily.dropna( subset = [ target_col ] ).copy( )
holdout_model = holdout_daily.dropna( subset = [ target_col ] ).copy( )

X_train = train_model[ all_feature_cols ].copy( )
X_holdout = holdout_model[ all_feature_cols ].copy( )
fill_values = X_train.median( numeric_only = True )
X_train = X_train.fillna( fill_values )
X_holdout = X_holdout.fillna( fill_values )
y_train = train_model[ target_col ]
y_holdout = holdout_model[ target_col ]

naive_by_doy = train_model.groupby( train_model[ 'date' ].dt.dayofyear )[ target_col ].median( )
holdout_model[ 'pred_climatology' ] = holdout_model[ 'date' ].dt.dayofyear.map( naive_by_doy )
holdout_model[ 'pred_climatology' ] = holdout_model[ 'pred_climatology' ].fillna( float( y_train.median( ) ) )

ridge_model = Ridge( alpha = 1.0 )
ridge_model.fit( X_train, y_train )
holdout_model[ 'pred_ridge' ] = ridge_model.predict( X_holdout )

hgb_model = HistGradientBoostingRegressor( learning_rate = 0.05, max_depth = 6, max_iter = 250, random_state = 42 )
hgb_model.fit( X_train, y_train )
holdout_model[ 'pred_hgb' ] = hgb_model.predict( X_holdout )

score_rows = [ ]
for model_name, pred_col in [ ( 'naive_climatology_doy', 'pred_climatology' ), ( 'ridge_air_context', 'pred_ridge' ), ( 'hgb_air_context', 'pred_hgb' ) ]:
    pred = holdout_model[ pred_col ]
    score_rows.append( { 
        'model': model_name,
        'mae': float( mean_absolute_error( y_holdout, pred ) ),
        'rmse': float( mean_squared_error( y_holdout, pred ) ** 0.5 ),
        'r2': float( r2_score( y_holdout, pred ) ),
        'n_holdout': int( len( holdout_model ) ),
    } )

phase6_scores = pd.DataFrame( score_rows ).sort_values( 'rmse' ).reset_index( drop = True )
selected_model_name = str( phase6_scores.iloc[ 0 ][ 'model' ] )
selected_model = hgb_model if selected_model_name == 'hgb_air_context' else ridge_model
selected_pred_col = 'pred_hgb' if selected_model_name == 'hgb_air_context' else 'pred_ridge'

print( 'phase 3 scores:' )
display( phase6_scores.round( 4 ) )
print( f'selected model: {selected_model_name}' )

In [ ]:
# Score the selected model on all screened rows.
# This file is what phase 4 will treat as fresh material.
all_model = daily.dropna( subset = [ target_col ] ).copy( )
all_model[ all_feature_cols ] = all_model[ all_feature_cols ].fillna( fill_values )
all_model[ 'delta_water_temp_pred' ] = selected_model.predict( all_model[ all_feature_cols ] )
all_model[ 'water_temp_pred' ] = all_model[ 'water_temp_baseline' ] + all_model[ 'delta_water_temp_pred' ]

fig, axes = plt.subplots( 1, 2, figsize = ( 15, 5 ) )

sns.scatterplot( data = holdout_model, x = target_col, y = selected_pred_col, s = 18, alpha = 0.5, ax = axes[ 0 ] )
lo = float( min( holdout_model[ target_col ].min( ), holdout_model[ selected_pred_col ].min( ) ) )
hi = float( max( holdout_model[ target_col ].max( ), holdout_model[ selected_pred_col ].max( ) ) )
axes[ 0 ].plot( [ lo, hi ], [ lo, hi ], color = 'black', linestyle = '--', linewidth = 1.0 )
axes[ 0 ].set_title( 'Phase 3 Holdout: Observed vs Predicted Delta Water Temp' )
axes[ 0 ].set_xlabel( 'Observed delta water temp' )
axes[ 0 ].set_ylabel( 'Predicted delta water temp' )

holdout_model[ 'residual' ] = holdout_model[ selected_pred_col ] - holdout_model[ target_col ]
sns.histplot( data = holdout_model, x = 'residual', bins = 40, ax = axes[ 1 ] )
axes[ 1 ].axvline( 0, color = 'black', linestyle = '--', linewidth = 1.0 )
axes[ 1 ].set_title( 'Phase 3 Holdout Residuals' )
axes[ 1 ].set_xlabel( 'Predicted - observed' )
axes[ 1 ].set_ylabel( 'Count' )

plt.tight_layout( )
plt.show( )

In [ ]:
phase6_scores.to_csv( scores_out_path, index = False )
all_model.to_csv( daily_pred_out_path, index = False )
holdout_model.to_csv( holdout_pred_out_path, index = False )

print( f'saved: {scores_out_path}' )
print( f'saved: {daily_pred_out_path}' )
print( f'saved: {holdout_pred_out_path}' )